# Synthetic Light-Curve Feature Pipeline

We use explicitly synthetic dual-image curves and toy fit outcomes to exercise RubinsCSSL's feature-extraction and population-summary pipeline. Curve sampling, overlap, photometric errors, and supplied fit assessments feed the band summary table and its static and interactive plots.

In [ ]:
from pathlib import Path
import os
import sys
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
repository_root = next(
    candidate for candidate in search_roots
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "src" / "rubinscssl").is_dir()
)
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
if str(repository_root.parent.parent / "tansudaylan" / "tdpy") not in sys.path:
    sys.path.insert(0, str(repository_root.parent.parent / "tansudaylan" / "tdpy"))

from rubinscssl.curve_features import build_population_analysis_tables
from rubinscssl.plot_utils import plot_band_performance, plot_band_performance_bokeh
from tdpy.verbosity import print

visuals_directory = repository_root / "visuals"
visuals_directory.mkdir(exist_ok=True)
rng = np.random.default_rng(42)

In [ ]:
start_jd = 2460000.0  # [day]
cadence_days = 2.0  # [day]
base_epochs = start_jd + np.arange(15) * cadence_days
bands = ("g", "r", "i")
lens_systems = {}
baseline_rows = []


def make_curve(epochs, magnitude_offset, uncertainty_mag, add_noise):
    phase = (epochs - start_jd) / 9.0
    magnitudes = 20.0 + magnitude_offset + 0.12 * np.sin(phase)
    if add_noise:
        magnitudes = magnitudes + rng.normal(0.0, uncertainty_mag, size=len(epochs))
    return SimpleNamespace(
        jds=np.asarray(epochs, dtype=float),
        mags=np.asarray(magnitudes, dtype=float),
        magerrs=np.full(len(epochs), uncertainty_mag, dtype=float),
    )


for system_index in range(6):
    system_id = f"toy-lens-{system_index + 1}"
    true_delay_days = 4.0 + system_index  # [day]
    system = {
        "system_status": "success",
        "batch": "synthetic-demo",
        "time_delay": [0.0, true_delay_days],  # [day]
        "z_lens": 0.4 + 0.02 * system_index,
        "z_source": 1.2 + 0.03 * system_index,
        "einstein_radius": 1.0 + 0.1 * system_index,  # [arcsec]
    }
    for band_index, band in enumerate(bands):
        epochs_a = base_epochs
        keep_b = np.ones(len(base_epochs), dtype=bool)
        keep_b[(system_index + band_index) % 4::5] = False
        epochs_b = base_epochs[keep_b]
        uncertainty_mag = 0.015 + 0.005 * band_index  # [mag]
        model_curves = [
            make_curve(epochs_a, 0.0, uncertainty_mag, False),
            make_curve(epochs_b, 0.2, uncertainty_mag, False),
        ]
        observed_curves = [
            make_curve(epochs_a, 0.0, uncertainty_mag, True),
            make_curve(epochs_b, 0.2, uncertainty_mag, True),
        ]
        system[band] = {
            "band_status": "success",
            "model_lcs": model_curves,
            "lcs": observed_curves,
        }
        for curve_type, offset in (("model", 0.05), ("observed", 0.15 + 0.08 * system_index)):
            estimated_delay_days = true_delay_days + offset  # [day]
            baseline_rows.append({
                "system_id": system_id,
                "band": band,
                "true_delay": true_delay_days,  # [day]
                f"{curve_type}_status": "success",
                f"{curve_type}_spline_results": {
                    "converged": True,
                    "delay": estimated_delay_days,  # [day]
                    "n_A": len(epochs_a),
                    "n_B": len(epochs_b),
                    "runtime_seconds": 0.0,  # [s], synthetic fixture value
                },
                f"{curve_type}_spline_assessment": {
                    "assessable": True,
                    "accurate": abs(offset) < 0.5,
                    "delay_error": offset,  # [day]
                    "absolute_delay_error": abs(offset),  # [day]
                    "relative_delay_error": abs(offset) / true_delay_days,
                    "tolerance": 0.5,  # [day], illustrative tolerance
                },
                "model_spline_assessment": {"accurate": True},
            })
    lens_systems[system_id] = system

print(f"Created explicitly synthetic curves for {len(lens_systems)} toy systems in {len(bands)} bands.")

In [ ]:
analysis_tables = build_population_analysis_tables(
    lens_systems,
    {"spline": baseline_rows},
)
feature_table = analysis_tables["features"]
analysis_table = analysis_tables["analysis"]
band_summary = analysis_tables["band_summary"]

print(f"Feature rows: {len(feature_table)}; population rows: {len(analysis_table)}.")
display(feature_table[[
    "system_id", "band", "curve_type", "feature_status",
    "n_min", "overlap_baseline", "median_cadence", "median_error",
]])
display(band_summary[[
    "band", "curve_type", "estimator", "n_rows", "n_attempted",
    "n_assessable", "n_accurate", "end_to_end_accurate_fraction",
]])

In [ ]:
png_path = visuals_directory / "rubinscssl_synthetic_band_performance.png"
print("Writing to visuals/rubinscssl_synthetic_band_performance.png...")
static_figure = plot_band_performance(
    band_summary,
    curve_type="observed",
    estimator="spline",
    filename=png_path,
    typeplotback="white",
)
display(static_figure)
plt.close(static_figure)

try:
    bokeh_relative_path = Path(os.path.relpath(
        visuals_directory / "rubinscssl_synthetic_band_performance.html",
        Path.cwd(),
    ))
    interactive_path = plot_band_performance_bokeh(
        band_summary,
        curve_type="observed",
        estimator="spline",
        filename=bokeh_relative_path,
        typeplotback="white",
    )
except ImportError:
    print("Install the optional plots extra to generate the Bokeh figure.")
else:
    print(f"Interactive figure saved under visuals/{interactive_path.name}.")